# Financial Machine Learning: Meta-Labeling for Trading Strategy Refinement
**Project**: Improving Base Regression Models using Meta-Labeling (Marcos López de Prado's Methodology).
**Goal**: Filter out false positive signals from the Base Model during high-risk market regimes to improve Risk-Adjusted Returns.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, precision_score

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Data Loading & Problem Definition
We start by loading the predictions generated by the Base Model (XGBoost).
The Base Model predicts the raw price of AAPL. We convert this to a directional signal.


In [2]:
# Load predictions
df = pd.read_csv('colleagues_predictions.csv', index_col=0, parse_dates=True)

# Generate Signals
df['Signal'] = np.where(df['Predicted_Close'] > df['Prev_Close'], 1, -1)
df['Actual_Direction'] = np.where(df['Actual_Close'] > df['Prev_Close'], 1, -1)

# Meta-Labeling Target: 1 if Base Model was correct, 0 otherwise
df['Meta_Target'] = (df['Signal'] == df['Actual_Direction']).astype(int)

print(f"Base Model Accuracy: {df['Meta_Target'].mean():.2%}")


Base Model Accuracy: 47.18%


## 2. Feature Engineering: Market Regime
The core hypothesis is that the Base Model fails during specific market regimes (e.g., high volatility).
We introduce "Meta-Features" to detect these regimes:
- **VIX_Slope**: Is fear rising?
- **BB_Width**: Is the market volatile?
- **RSI_Dist**: Is the trend overextended?
- **Model_Confidence**: How confident is the Base Model?


In [3]:
# Calculate Meta-Features (already in CSV from step4, but let's visualize)
meta_features = ['RSI', 'ATR', 'Rolling_Std', 'Model_Confidence', 'VIX_Slope', 'BB_Width', 'RSI_Dist']

# Correlation Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(df[meta_features].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation of Meta-Features')
plt.show()


KeyError: "['Model_Confidence'] not in index"

<Figure size 1000x800 with 0 Axes>

## 3. Meta-Model Training
We train a Random Forest Classifier to predict `Meta_Target` using the Meta-Features.
We use `GridSearchCV` to optimize for Precision (we want to be sure when we trade).


In [ ]:
X_meta = df[meta_features]
y_meta = df['Meta_Target']

# Split Data (50/50)
split = int(len(df) * 0.5)
X_train, X_test = X_meta.iloc[:split], X_meta.iloc[split:]
y_train, y_test = y_meta.iloc[:split], y_meta.iloc[split:]

# Train Optimized Model (Parameters from Step 5)
rf = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_leaf=3, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Predict
meta_preds = rf.predict(X_test)

print("Meta-Model Performance:")
print(classification_report(y_test, meta_preds))


## 4. Results & Equity Curve
We simulate the trading strategy:
- **Base Strategy**: Trade every signal from the Base Model.
- **Meta Strategy**: Trade only when the Meta-Model predicts "1" (High Probability of Success).


In [ ]:
# Simulation
test_data = df.iloc[split:].copy()
test_data['Meta_Filter'] = meta_preds

# Calculate Returns
test_data['Return'] = (test_data['Actual_Close'] - test_data['Prev_Close']) / test_data['Prev_Close']
test_data['Strategy_Base'] = test_data['Return'] * test_data['Signal']
test_data['Strategy_Meta'] = test_data['Strategy_Base'] * test_data['Meta_Filter']

# Cumulative Returns
test_data['Equity_Base'] = (1 + test_data['Strategy_Base']).cumprod()
test_data['Equity_Meta'] = (1 + test_data['Strategy_Meta']).cumprod()

# Plot
plt.figure(figsize=(12, 6))
plt.plot(test_data['Equity_Base'], label='Base Strategy (Regression Only)', color='red', alpha=0.6)
plt.plot(test_data['Equity_Meta'], label='Meta-Labeling Strategy (Hybrid)', color='green', linewidth=2)
plt.title('Equity Curve Comparison')
plt.legend()
plt.show()


## 5. Explainability: Why it works?
Which features are most important for the filter?


In [ ]:
importances = pd.Series(rf.feature_importances_, index=meta_features)
importances.sort_values().plot(kind='barh', color='teal')
plt.title('Feature Importance for Meta-Model')
plt.show()
